# Atividade Prática: Visualização Geoespacial com Folium

## Contexto do Problema

Desenvolvimento de um módulo de inteligência geográfica para mapear oportunidades imobiliárias e analisar a distribuição de propriedades nos municípios de Nova Iguaçu e Queimados. O objetivo é criar um mapa interativo que permita visualizar a localização exata das propriedades, identificar concentrações de ofertas através de clusters e analisar o valor de mercado de forma espacial.

## 1. Configuração do Ambiente e Base de Dados

Importação das bibliotecas e geração do DataFrame com as coordenadas geográficas simuladas.

In [ ]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

### Geração dos dados sintéticos

Criação de 45 imóveis distribuídos entre Nova Iguaçu e Queimados, com valor de venda e tipo. As coordenadas são atribuídas com uma pequena dispersão aleatória em torno do centro de cada cidade.

In [ ]:
np.random.seed(42)
n_imoveis = 45

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}

df_mapa = pd.DataFrame(dados_imoveis)

### Atribuição de coordenadas

Geração da latitude e longitude com base na cidade, adicionando uma pequena dispersão aleatória.

In [ ]:
def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)

df_mapa.head()

## 2. Roteiro de Tarefas

## Parte 1: Inicialização e Marcadores Básicos

### Tarefa 1: Mapa base centralizado

Criação de um mapa base com `folium.Map` centralizado na coordenada média de todos os imóveis, com `zoom_start=12` e estilo padrão OpenStreetMap.

In [ ]:
lat_media = df_mapa['latitude'].mean()
lon_media = df_mapa['longitude'].mean()

mapa1 = folium.Map(location=[lat_media, lon_media], zoom_start=12)
mapa1

### Tarefa 2: Marcadores simples

Iteração sobre as 5 primeiras linhas do DataFrame, adicionando um `folium.Marker` para cada imóvel. O popup exibe o tipo e o valor de venda.

In [ ]:
for _, linha in df_mapa.head(5).iterrows():
    folium.Marker(
        location=[linha['latitude'], linha['longitude']],
        popup=f"{linha['tipo']} — R$ {linha['valor_venda']:,.2f}"
    ).add_to(mapa1)

mapa1

## Parte 2: Customização Visual com Marcadores Circulares

### Tarefa 3: Novo mapa base com `CircleMarker`

Criação de um novo mapa base para representar todos os imóveis com `folium.CircleMarker`.

In [ ]:
mapa2 = folium.Map(location=[lat_media, lon_media], zoom_start=12)
mapa2

### Tarefa 4: Regras de negócio dos marcadores

Configuração dos círculos com:
- **Raio:** 8 pixels
- **Cor:** azul para Nova Iguaçu, laranja para Queimados
- **Tooltip:** "Clique para detalhes"

In [ ]:
for _, linha in df_mapa.iterrows():
    cor = 'blue' if linha['cidade'] == 'Nova Iguaçu' else 'orange'

    folium.CircleMarker(
        location=[linha['latitude'], linha['longitude']],
        radius=8,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.7,
        tooltip='Clique para detalhes'
    ).add_to(mapa2)

mapa2

## Parte 3: Agrupamento Inteligente (Clustering)

### Tarefa 5: Mapa com `MarkerCluster`

Criação de um terceiro mapa base e instanciação de um objeto `MarkerCluster()` para agrupar pontos próximos e evitar poluição visual.

In [ ]:
mapa3 = folium.Map(location=[lat_media, lon_media], zoom_start=12)
cluster = MarkerCluster().add_to(mapa3)

### Tarefa 6: Adição dos imóveis ao cluster

Adição de todos os imóveis ao cluster com ícones customizados (`folium.Icon`), variando a cor conforme o tipo: verde para Casa, azul para Apartamento, cinza para Terreno.

In [ ]:
cores_tipo = {
    'Casa': 'green',
    'Apartamento': 'blue',
    'Terreno': 'gray'
}

for _, linha in df_mapa.iterrows():
    folium.Marker(
        location=[linha['latitude'], linha['longitude']],
        popup=f"{linha['tipo']} — R$ {linha['valor_venda']:,.2f}",
        icon=folium.Icon(color=cores_tipo[linha['tipo']])
    ).add_to(cluster)

mapa3

### Tarefa 7: Salvamento do mapa final

Salvamento do mapa final em um arquivo HTML chamado `mapa_imoveis_baixada.html`.

In [ ]:
mapa3.save('mapa_imoveis_baixada.html')
print('Mapa salvo em mapa_imoveis_baixada.html')